# GeoCebada — cobertura y alineación temporal

Esta notebook estudia cuándo tenemos observaciones satelitales para cada parcela y transforma las capturas irregulares de BASIC/PRO a una rejilla temporal común.

La idea es: `capturas irregulares → k vecinos temporales → curva alineada → features comparables entre parcelas`.

No se usan los targets ocultos de `PREDICCION`.

> Sabemos que el target corresponde al ciclo abril–octubre de 2025, pero el cutoff operacional exacto para una predicción pre-cosecha todavía debe fijarse. Usar todo abril–octubre aquí es exploratorio.


## 0. Entorno

La celda siguiente encuentra el root e instala GeoCebada en modo editable. En una computadora normal deja `INSTALL_GPU_EXTRA = False`. En el servidor NVIDIA puedes cambiarlo a `True` para instalar CuPy/CUDA 12.


In [ ]:
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "geocebada").exists():
            return candidate
    raise RuntimeError("No se encontró la raíz de GeoCebada. Abre Jupyter dentro del repositorio.")


ROOT = find_repo_root()
INSTALL_GPU_EXTRA = False  # True sólo en una máquina NVIDIA/CUDA compatible.
extras = "dev,geo,gpu" if INSTALL_GPU_EXTRA else "dev,geo"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[{extras}]"])

print(f"Repositorio: {ROOT}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from geocebada.data import attach_yield_split_metadata, load_basic_data, load_pro_data, load_yield_split
from geocebada.features import (
    align_temporal_knn,
    aligned_to_wide,
    make_temporal_grid,
    temporal_backend_available,
    temporal_coverage_summary,
)

pd.set_option("display.max_columns", 30)
print("Backend temporal detectado:", temporal_backend_available("auto"))


## 1. Configuración

Para el primer experimento usamos sólo `*_promedio`. Después puedes poner `USE_ALL_INDEX_STATS = True` para alinear también `std`, `min` y `max`.

- `GRID_FREQ="14D"`: rejilla quincenal aproximada.
- `K=3`: tres capturas más cercanas por fecha objetivo.
- `BANDWIDTH_DAYS=14`: decaimiento exponencial con la distancia temporal.
- `MAX_DISTANCE_DAYS=30`: si no hay captura suficientemente cercana, se deja `NaN`.
- `CLOUD_WEIGHTING=True`: reduce el peso de observaciones nubladas.
- `BACKEND="auto"`: GPU si CuPy/CUDA está disponible; CPU en caso contrario.


In [ ]:
SOURCE = "BASIC"  # BASIC o PRO
START_DATE = "2025-04-01"
END_DATE = "2025-10-31"  # exploratorio hasta fijar el cutoff operacional
GRID_FREQ = "14D"
K = 3
BANDWIDTH_DAYS = 14.0
MAX_DISTANCE_DAYS = 30.0
CLOUD_WEIGHTING = True
BACKEND = "auto"
USE_ALL_INDEX_STATS = False
RANDOM_SEED = 42


## 2. Cargar BASIC o PRO

Estas tablas son longitudinales: una fila es aproximadamente `parcela × fecha × sensor`, no una muestra independiente de rendimiento.


In [ ]:
if SOURCE.upper() == "BASIC":
    data = load_basic_data(attach_split=True)
elif SOURCE.upper() == "PRO":
    data = load_pro_data(attach_split=True)
else:
    raise ValueError("SOURCE debe ser 'BASIC' o 'PRO'.")

data["fecha_captura"] = pd.to_datetime(data["fecha_captura"], errors="coerce")
print("Shape:", data.shape)
print("Parcelas:", data["ID_POLIGONO"].nunique())
print("Sensores:", sorted(data["sensor"].dropna().astype(str).unique()))
print("Fechas:", data["fecha_captura"].min(), "→", data["fecha_captura"].max())
display(data.head())


## 3. Cobertura temporal: entrenamiento vs predicción

Esto sólo inspecciona covariables: disponibilidad, gaps y nubosidad. No usa el rendimiento oculto.


In [ ]:
coverage = temporal_coverage_summary(data, start=START_DATE, end=END_DATE)
split = load_yield_split()[["ID_POLIGONO", "CONJUNTO"]]
coverage = coverage.merge(split, on="ID_POLIGONO", how="left", validate="many_to_one")
display(coverage.head())

summary_columns = [
    column
    for column in ["n_records", "n_unique_dates", "median_gap_days", "max_gap_days", "cloud_mean"]
    if column in coverage.columns
]
display(
    coverage.groupby(["CONJUNTO", "sensor"], dropna=False)[summary_columns]
    .agg(["mean", "median", "std"])
    .round(2)
)


### Mapa de disponibilidad por semana

Cada celda indica si una parcela tuvo al menos una captura durante esa semana. Sirve para detectar huecos temporales de forma visual.


In [ ]:
period = data.loc[
    data["fecha_captura"].between(START_DATE, END_DATE, inclusive="both"),
    ["ID_POLIGONO", "fecha_captura"],
].dropna()
period["week"] = period["fecha_captura"].dt.to_period("W").astype(str)
availability = (pd.crosstab(period["ID_POLIGONO"], period["week"]) > 0).astype(int)
order = split.assign(
    _order=split["CONJUNTO"].map({"ENTRENAMIENTO": 0, "PREDICCION": 1})
).sort_values(["_order", "ID_POLIGONO"])["ID_POLIGONO"]
availability = availability.reindex(order).fillna(0)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(availability.to_numpy(), aspect="auto", interpolation="nearest")
ax.set_title(f"Disponibilidad semanal — {SOURCE}")
ax.set_xlabel("Semana")
ax.set_ylabel("Parcelas")
step = max(1, availability.shape[1] // 12)
ticks = np.arange(0, availability.shape[1], step)
ax.set_xticks(ticks)
ax.set_xticklabels(availability.columns[ticks], rotation=60, ha="right")
plt.tight_layout()
plt.show()


## 4. Variables a alinear


In [ ]:
metadata = {
    "ID_POLIGONO",
    "fecha_captura",
    "sensor",
    "porcentaje_nubosidad",
    "AREA_HA",
    "RENDIMIENTO_T_HA",
    "CONJUNTO",
}
numeric_candidates = [
    column for column in data.select_dtypes(include="number").columns if column not in metadata
]
VALUE_COLUMNS = (
    numeric_candidates
    if USE_ALL_INDEX_STATS
    else [column for column in numeric_candidates if column.endswith("_promedio")]
)
print(f"Variables a alinear: {len(VALUE_COLUMNS)}")
print(VALUE_COLUMNS)


## 5. Alineación k-NN temporal

Para cada fecha de la rejilla se toman las `K` capturas más cercanas de la misma parcela y sensor. La contribución cae exponencialmente con la distancia en días y puede penalizarse por nubosidad.


In [ ]:
grid = make_temporal_grid(START_DATE, END_DATE, freq=GRID_FREQ)
print("Puntos de rejilla:", len(grid))
print(list(grid.strftime("%Y-%m-%d")))


In [ ]:
start_time = time.perf_counter()
aligned = align_temporal_knn(
    data,
    VALUE_COLUMNS,
    grid=grid,
    start=START_DATE,
    end=END_DATE,
    k=K,
    bandwidth_days=BANDWIDTH_DAYS,
    max_distance_days=MAX_DISTANCE_DAYS,
    cloud_weighting=CLOUD_WEIGHTING,
    backend=BACKEND,
)
elapsed = time.perf_counter() - start_time
print("Backend usado:", aligned["backend"].iloc[0] if len(aligned) else "sin filas")
print(f"Tiempo: {elapsed:.2f} s")
print("Shape alineado:", aligned.shape)
display(aligned.head())


### Diagnóstico de gaps y faltantes


In [ ]:
display(
    aligned[["neighbor_count", "nearest_gap_days"]]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)
display(
    aligned[VALUE_COLUMNS]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
    .head(30)
)


## 6. Curvas alineadas de algunas parcelas

Los sensores se mantienen separados para no fusionar automáticamente mediciones que pueden tener distinta calibración.


In [ ]:
plot_variable = "ndvi_promedio" if "ndvi_promedio" in aligned.columns else VALUE_COLUMNS[0]
rng = np.random.default_rng(RANDOM_SEED)
parcel_ids = aligned["ID_POLIGONO"].drop_duplicates().to_numpy()
sample_ids = rng.choice(parcel_ids, size=min(8, len(parcel_ids)), replace=False)

fig, ax = plt.subplots(figsize=(13, 6))
for (parcel_id, sensor), group in aligned.loc[
    aligned["ID_POLIGONO"].isin(sample_ids)
].groupby(["ID_POLIGONO", "sensor"], dropna=False):
    ax.plot(
        group["grid_date"],
        group[plot_variable],
        marker="o",
        linewidth=1,
        markersize=3,
        label=f"{parcel_id} · {sensor}",
    )
ax.set_title(f"{plot_variable}: curvas alineadas")
ax.set_xlabel("Fecha")
ax.set_ylabel(plot_variable)
ax.legend(ncol=2, fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 7. Una fila por parcela

Esta tabla ya puede entrar a modelos tabulares posteriores. El sensor queda incluido en cada nombre de feature.


In [ ]:
wide = aligned_to_wide(aligned, value_columns=VALUE_COLUMNS)
wide = attach_yield_split_metadata(wide)
print("Shape model-ready temporal:", wide.shape)
print("Parcelas:", wide["ID_POLIGONO"].nunique())
print("Entrenamiento:", (wide["CONJUNTO"] == "ENTRENAMIENTO").sum())
print("Predicción:", (wide["CONJUNTO"] == "PREDICCION").sum())
display(wide.iloc[:5, :20])


In [ ]:
temporal_features = [
    column
    for column in wide.columns
    if column not in {"ID_POLIGONO", "AREA_HA", "RENDIMIENTO_T_HA", "CONJUNTO"}
]
missing_by_split = (
    wide.groupby("CONJUNTO")[temporal_features]
    .apply(lambda group: group.isna().mean().mean() * 100)
    .rename("mean_feature_missing_pct")
)
display(missing_by_split.to_frame())


## 8. Guardar resultado opcional

Los datos derivados van en `data/interim/`, nunca en `data/source/`.


In [ ]:
SAVE_INTERIM = False
if SAVE_INTERIM:
    output_dir = ROOT / "data" / "interim"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"temporal_aligned_{SOURCE.lower()}_2025.parquet"
    wide.to_parquet(output_path, index=False)
    print("Guardado:", output_path)
else:
    print("No se guardó archivo. Cambia SAVE_INTERIM=True para exportar.")


## 9. CPU vs GPU

La RTX 4500 Ada puede acelerar las operaciones matriciales con CuPy, pero Pandas y el agrupamiento por parcela/sensor siguen en CPU. Con ~100 mil filas, el i9 y 128 GB RAM ya deberían ser muy rápidos. Para medir de verdad, corre exactamente el mismo bloque con `BACKEND="cpu"` y luego `BACKEND="gpu"`.

La GPU será más útil al alinear todas las estadísticas, repetir muchos experimentos o entrenar modelos con soporte CUDA.


## 10. Diccionario de variables


In [ ]:
variables_path = ROOT / "docs" / "VARIABLES.md"
display(Markdown(variables_path.read_text(encoding="utf-8")))
